<a href="https://colab.research.google.com/github/robertrichard86/Kaggle/blob/main/Titanic2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Carregamento e Preparação dos Dados
Nesta etapa, importamos as bibliotecas essenciais (pandas, numpy, sklearn) e carregamos os datasets de treino e teste. Armazenamos o PassengerId do conjunto de teste separadamente para garantir que a submissão final esteja no formato correto exigido pelo Kaggle.

In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import re

#Dados do Kaggle: https://www.kaggle.com/competitions/titanic/data

train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

test_passenger_id = test_data['PassengerId']

#2. Engenharia de Atributos (Feature Engineering)
Aqui aplicamos o conhecimento do domínio sobre o desastre do Titanic para criar novas variáveis:

Title: Extraímos títulos (Mr, Miss, Master) dos nomes para identificar status social e faixas etárias implícitas.

FamilySize: Combinamos SibSp (irmãos/cônjuges) e Parch (pais/filhos) para entender a dinâmica de grupos.

IsAlone: Criamos uma variável binária para identificar passageiros sem acompanhantes.

Deck: Extraímos a letra da cabine, que indica a localização física do passageiro no navio

In [10]:
def extract_features(df):
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace('Mlle', 'Miss').replace('Ms', 'Miss').replace('Mme', 'Mrs')
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = 0
    df.loc[df['FamilySize'] == 1, 'IsAlone'] = 1
    df['Cabin'] = df['Cabin'].fillna('U')
    df['Deck'] = df['Cabin'].map(lambda x: re.compile("([a-zA-Z]+)").search(x).group())
    return df

train_data = extract_features(train_data)
test_data = extract_features(test_data)

<>:4: SyntaxWarning: invalid escape sequence '\.'
<>:4: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_173/3640759473.py:4: SyntaxWarning: invalid escape sequence '\.'
  df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


#3. Tratamento de Valores Nulos (Imputação Inteligente)
Em vez de usar uma média global, realizamos uma imputação mais refinada:

Age: Preenchemos as idades faltantes com base na mediana de cada Título (ex: a mediana de um 'Master' é muito menor que a de um 'Mr').

Embarked & Fare: Preenchemos as lacunas restantes com a moda e a mediana, respectivamente, garantindo que o modelo não receba valores vazios.

In [11]:
train_data['Age'] = train_data['Age'].fillna(train_data.groupby('Title')['Age'].transform('median'))
test_data['Age'] = test_data['Age'].fillna(test_data.groupby('Title')['Age'].transform('median'))

train_data['Embarked'].fillna(train_data['Embarked'].mode()[0], inplace=True)
test_data['Fare'].fillna(test_data['Fare'].median(), inplace=True)

/tmp/ipykernel_173/3626505660.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_data['Embarked'].fillna(train_data['Embarked'].mode()[0], inplace=True)
/tmp/ipykernel_173/3626505660.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(

#4. Pré-processamento e Encoding
Transformamos os dados para o formato matemático aceito pelo modelo:

One-Hot Encoding: Convertemos variáveis categóricas (Sex, Embarked, Title, Deck) em colunas binárias (0 e 1).

Alinhamento de Colunas: Garantimos que o conjunto de teste tenha exatamente as mesmas colunas que o conjunto de treino, preenchendo com 0 onde houver divergência.

In [12]:
cols_to_drop = ['Name', 'Ticket', 'Cabin', 'PassengerId']
train_data.drop(columns=cols_to_drop, inplace=True)
test_data.drop(columns=cols_to_drop, inplace=True)

categorical_cols = ['Sex', 'Embarked', 'Pclass', 'Title', 'Deck']
train_data = pd.get_dummies(train_data, columns=categorical_cols, drop_first=True)
test_data = pd.get_dummies(test_data, columns=categorical_cols, drop_first=True)

features = train_data.drop(columns=['Survived']).columns
test_data = test_data.reindex(columns=features, fill_value=0)

X = train_data[features]
y = train_data['Survived']

#5. Modelagem com Random Forest e Validação Cruzada
Configuramos o RandomForestClassifier com hiperparâmetros ajustados (max_depth=7) para evitar o overfitting (quando o modelo decora o treino mas erra no teste). Utilizamos a Validação Cruzada (K-Fold) para obter uma estimativa realista da acurácia do modelo em dados não vistos.

In [13]:
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=7,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

scores = cross_val_score(model, X, y, cv=5)
print(f"Acurácia Média (CV): {scores.mean():.4f} (+/- {scores.std():.4f})")

model.fit(X, y)

Acurácia Média (CV): 0.8260 (+/- 0.0170)


RandomForestClassifier(max_depth=7, min_samples_leaf=3, n_estimators=500,
                       n_jobs=-1, random_state=42)

#6. Geração do Arquivo de Submissão
Após o treino final com todos os dados disponíveis, realizamos a predição no conjunto de teste. Os resultados são estruturados em um DataFrame e exportados para o arquivo submission.csv, prontos para serem enviados à plataforma do Kaggle.

In [14]:
test_predictions = model.predict(test_data)
output = pd.DataFrame({'PassengerId': test_passenger_id, 'Survived': test_predictions})
output.to_csv('submission.csv', index=False)

print("Arquivo 'submission.csv' gerado com sucesso!")

Arquivo 'submission.csv' gerado com sucesso!
